# Soluciones · 01 Coeficiente de reflexión y carta de Smith

Soluciones resueltas de los ejercicios del [cuaderno 01](../../notebooks/01_reflexion_smith.ipynb). Intenta antes cada ejercicio por tu cuenta, sobre todo los circuitos misteriosos.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q git+https://github.com/telmomm/nanovna-measurement-course

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from vnacourse import plotting
from vnacourse.puzzles import mystery_trace
from vnacourse.reflection import gamma_to_z, z_to_gamma

plotting.use_course_style()

Z0 = 50.0
f_hz = np.linspace(10e6, 1.45e9, 600)
omega = 2 * np.pi * f_hz

## 1. Ida y vuelta

A mano, para $Z = 20 - j30\ \Omega$:

$$\Gamma = \frac{(20 - j30) - 50}{(20 - j30) + 50} = \frac{-30 - j30}{70 - j30}
= \frac{(-30 - j30)(70 + j30)}{70^2 + 30^2} = \frac{-1200 - j3000}{5800} = -0.207 - j0.517$$

In [ ]:
gamma = z_to_gamma(20 - 30j)
print(f"Γ = {gamma:.3f},  |Γ| = {abs(gamma):.3f},  ángulo = {np.degrees(np.angle(gamma)):.1f}°")
print(f"De vuelta a impedancia: {gamma_to_z(gamma):.1f} Ω")

gamma_2 = 0.5 * np.exp(1j * np.radians(90))
print(f"\nΓ = 0.5∠90°  →  Z = {gamma_to_z(gamma_2):.1f} Ω")

El punto está en la parte inferior izquierda de la carta: resistencia por debajo de 50 Ω (izquierda) y capacitivo (mitad inferior). El segundo punto, $\Gamma = j0.5$, es $Z = 30 + j40\ \Omega$: justo encima del centro *no* significa 50 Ω de resistencia. La circunferencia de 50 Ω pasa por el centro pero se curva hacia la derecha.

## 2. Admitancia

Toma impedancias cualesquiera $Z$ y compara $\Gamma(Z)$ con $\Gamma(Z_0^2/Z)$.

In [ ]:
rng = np.random.default_rng(1)
z = rng.uniform(1, 300, 5) + 1j * rng.uniform(-300, 300, 5)

for z_k in z:
    mirrored = z_to_gamma(Z0**2 / z_k)
    print(f"Z = {z_k:7.1f} Ω   Γ(Z) = {z_to_gamma(z_k):+.3f}   Γ(Z0²/Z) = {mirrored:+.3f}")

assert np.allclose(z_to_gamma(Z0**2 / z), -z_to_gamma(z))

Siempre opuestos. Algebraicamente, con $Z' = Z_0^2/Z$:

$$\Gamma' = \frac{Z_0^2/Z - Z_0}{Z_0^2/Z + Z_0} = \frac{Z_0 - Z}{Z_0 + Z} = -\Gamma$$

La impedancia normalizada de $Z'$ es $Z'/Z_0 = Z_0/Z$, que es la **admitancia** normalizada de $Z$. Así que girar un punto 180° en torno al centro convierte «leer impedancia» en «leer admitancia». Una línea de conductancia constante es, por tanto, la imagen por ese giro de una línea de resistencia constante: las circunferencias de conductancia son las de resistencia reflejadas a través del centro, tangentes en el cortocircuito en lugar de en el abierto.

## 3. Resonancia paralelo

$\dfrac{1}{Z} = \dfrac{1}{R} + \dfrac{1}{j\omega L} + j\omega C$, con la resonancia de nuevo en $f_0 = 1/(2\pi\sqrt{LC})$.

In [ ]:
R, L, C = 100.0, 5e-9, 20e-12
z_parallel = 1 / (1 / R + 1 / (1j * omega * L) + 1j * omega * C)
f0 = 1 / (2 * np.pi * np.sqrt(L * C))
gamma = z_to_gamma(z_parallel)

fig, ax = plt.subplots(figsize=(5.4, 5.4))
plotting.gamma_plane(ax, title="RLC paralelo: 100 Ω ∥ 5 nH ∥ 20 pF")
ax.plot(gamma.real, gamma.imag, label="traza, de 10 MHz a 1.45 GHz")
ax.plot(gamma[0].real, gamma[0].imag, "o", color=plotting.SERIES[0])
ax.plot(z_to_gamma(R).real, 0, "s", color=plotting.INK)
ax.annotate("$f_0$", (z_to_gamma(R).real, 0), xytext=(8, 8), textcoords="offset points")
ax.legend(loc="upper left", bbox_to_anchor=(0.0, 0.0))

below = f_hz < f0
print(f"Frecuencia de resonancia: {f0 / 1e6:.0f} MHz")
print(
    f"Reactancia positiva (inductiva) por debajo de la resonancia: "
    f"{np.all(z_parallel.imag[below] > 0)}"
)
print(
    f"Reactancia negativa (capacitiva) por encima de la resonancia: "
    f"{np.all(z_parallel.imag[~below] < 0)}"
)

En resonancia la bobina y el condensador se anulan como *admitancias* y solo queda la resistencia: la traza cruza el eje horizontal en el punto de 100 Ω. La traza sigue la circunferencia de **conductancia** de 100 Ω.

Por debajo de la resonancia el circuito es **inductivo** (mitad superior) y por encima es **capacitivo**: lo contrario que el circuito serie. En paralelo domina el elemento de *menor* impedancia, y a baja frecuencia ese es la bobina. La traza sigue girando en sentido horario.

## 4. Los circuitos misteriosos

El método es el mismo para los tres: mira la forma para decidir la topología y después lee los valores en la impedancia (serie) o en la admitancia (paralelo).

### Circuito A

La traza empieza cerca del abierto, se queda en la mitad inferior y sigue una circunferencia de resistencia: una resistencia **en serie con un condensador**. Entonces $\operatorname{Re} Z = R$ e $\operatorname{Im} Z = -1/(\omega C)$.

In [ ]:
z_a = gamma_to_z(mystery_trace("A", f_hz))
r_a = z_a.real
c_a = -1 / (omega * z_a.imag)
print(f"Re Z: de {r_a.min():.2f} a {r_a.max():.2f} Ω  →  R = {r_a.mean():.0f} Ω")
print(f"C en cada punto de frecuencia: de {c_a.min() * 1e12:.2f} a {c_a.max() * 1e12:.2f} pF")

**A es 30 Ω en serie con 8 pF.** Que $R$ y $C$ salgan iguales a todas las frecuencias confirma la topología: con el modelo equivocado irían cambiando.

### Circuito B

La traza empieza en el cortocircuito, sube por la mitad superior siguiendo una circunferencia tangente en el cortocircuito y se dirige a un punto a la derecha del eje horizontal: una resistencia **en paralelo con una bobina**. Trabaja con la admitancia $Y = 1/Z = 1/R + 1/(j\omega L)$.

In [ ]:
y_b = 1 / gamma_to_z(mystery_trace("B", f_hz))
r_b = 1 / y_b.real
l_b = -1 / (omega * y_b.imag)
print(f"1 / Re Y: de {r_b.min():.1f} a {r_b.max():.1f} Ω  →  R = {r_b.mean():.0f} Ω")
print(f"L en cada punto de frecuencia: de {l_b.min() * 1e9:.2f} a {l_b.max() * 1e9:.2f} nH")

**B es 120 Ω en paralelo con 15 nH.**

### Circuito C

La traza sigue una circunferencia de resistencia de la mitad inferior a la superior y cruza el eje horizontal: un **RLC serie**. La resistencia es $\operatorname{Re} Z$, la frecuencia de resonancia es donde $\operatorname{Im} Z = 0$, y la reactancia $X = \omega L - \dfrac{1}{\omega C}$ es lineal en las dos incógnitas $L$ y $1/C$, así que un ajuste por mínimos cuadrados en todas las frecuencias da ambas.

In [ ]:
z_c = gamma_to_z(mystery_trace("C", f_hz))
r_c = z_c.real.mean()

crossing = np.flatnonzero(np.diff(np.sign(z_c.imag)))[0]
f0_c = np.interp(0, z_c.imag[crossing : crossing + 2], f_hz[crossing : crossing + 2])

# Escala las dos columnas a tamaños parecidos (ω es ~1e9, 1/ω es ~1e-10) para que el ajuste
# esté bien condicionado.
omega_ref = 2 * np.pi * 1e9
design = np.column_stack([omega / omega_ref, -omega_ref / omega])
(a, b), *_ = np.linalg.lstsq(design, z_c.imag, rcond=None)
l_c, inverse_c = a / omega_ref, b * omega_ref

print(f"R = {r_c:.0f} Ω")
print(f"Resonancia (Im Z = 0) a {f0_c / 1e6:.0f} MHz")
print(f"L = {l_c * 1e9:.1f} nH,  C = {1 / inverse_c * 1e12:.2f} pF")
print(f"Comprobación: 1 / (2π√(LC)) = {1 / (2 * np.pi * np.sqrt(l_c / inverse_c)) / 1e6:.0f} MHz")

**C es 60 Ω en serie con 30 nH y 2 pF**, con resonancia a 650 MHz.

Conocer solo $f_0$ daría únicamente el producto $LC$. La segunda información es lo rápido que cambia la reactancia en torno a la resonancia, que es lo que usa el ajuste. Con datos reales y ruidosos harías lo mismo, y el residuo del ajuste te diría lo bien que el modelo de tres elementos describe el dispositivo.